In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# พารามิเตอร์ที่ใช้บ่อยสำหรับการ transpile

<details>
<summary><b>Package versions</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
หน้านี้อธิบายพารามิเตอร์ที่ใช้บ่อยกว่าสำหรับการ transpile ในเครื่อง พารามิเตอร์เหล่านี้ตั้งค่าผ่าน arguments ของ [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) หรือ [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile)

<span id="approx-degree"></span>
## ระดับการประมาณค่า
คุณสามารถใช้ approximation degree เพื่อระบุว่าต้องการให้ circuit ผลลัพธ์ตรงกับ circuit ที่ต้องการ (input) มากแค่ไหน ค่านี้เป็น float ในช่วง (0.0 - 1.0) โดย 0.0 คือการประมาณค่าสูงสุด และ 1.0 (ค่าเริ่มต้น) คือไม่มีการประมาณค่า ค่าที่น้อยกว่าจะแลกความแม่นยำของ output กับความสะดวกในการรัน (นั่นคือ Gate น้อยลง) ค่าเริ่มต้นคือ 1.0

ในการสังเคราะห์ unitary สอง-Qubit (ใช้ในขั้นตอนเริ่มต้นของทุกระดับและสำหรับขั้นตอน optimization ที่ optimization level 3) ค่านี้ระบุ target fidelity ของการ decompose ผลลัพธ์ กล่าวคือ ข้อผิดพลาดเท่าไรที่เกิดขึ้นเมื่อ matrix representation ของ circuit ถูกแปลงเป็น discrete gates หาก approximation degree มีค่าน้อยกว่า (ประมาณมากกว่า) circuit ผลลัพธ์จากการสังเคราะห์จะต่างจาก input matrix มากกว่า แต่ก็มีแนวโน้มที่จะมี Gate น้อยกว่า (เนื่องจาก arbitrary two-qubit operation ใดๆ สามารถ decompose ได้อย่างสมบูรณ์ด้วย CX gate สูงสุดสามตัว) และรันได้ง่ายกว่า

เมื่อ approximation degree น้อยกว่า 1.0 Circuit ที่มี CX gate หนึ่งหรือสองตัวอาจถูกสังเคราะห์ ทำให้ข้อผิดพลาดจากฮาร์ดแวร์น้อยลง แต่ข้อผิดพลาดจากการประมาณค่าเพิ่มขึ้น เนื่องจาก CX เป็น Gate ที่มีราคาแพงที่สุดในแง่ของข้อผิดพลาด จึงอาจเป็นประโยชน์ที่จะลดจำนวนลงโดยแลกกับ fidelity ในการสังเคราะห์ (เทคนิคนี้ถูกใช้เพื่อเพิ่ม quantum volume บนอุปกรณ์ IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926))

ตัวอย่างเช่น เราสร้าง `UnitaryGate` สอง-Qubit แบบสุ่มซึ่งจะถูกสังเคราะห์ในขั้นตอนเริ่มต้น การตั้งค่า `approximation_degree` น้อยกว่า 1.0 อาจสร้าง circuit ที่เป็นการประมาณค่า นอกจากนี้เราต้องระบุ `basis_gates` เพื่อให้วิธีการสังเคราะห์รู้ว่า Gate ไหนที่สามารถใช้สำหรับการสังเคราะห์แบบประมาณค่า

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

ผลลัพธ์คือ `2` เนื่องจากการประมาณค่าต้องใช้ CX gate น้อยกว่า

<span id="seed"></span>
## Seed ของตัวสร้างตัวเลขสุ่ม
บางส่วนของ Transpiler เป็น stochastic ดังนั้นการรัน transpile ซ้ำๆ อาจคืนผลลัพธ์ที่แตกต่างกัน เพื่อให้ได้ผลลัพธ์ที่ reproducible คุณสามารถตั้งค่า seed สำหรับ pseudorandom number generator โดยใช้ argument `seed_transpiler` การรันซ้ำๆ โดยใช้ seed เดียวกันจะคืนผลลัพธ์เดียวกัน

ตัวอย่าง:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Layout เริ่มต้น
ก่อนการ transpile Qubit ที่อยู่ใน Circuit ของคุณเป็น virtual qubit ที่ไม่จำเป็นต้องตรงกับ physical qubit บน Backend เป้าหมาย คุณสามารถระบุการ mapping เริ่มต้นของ virtual qubit ไปยัง physical qubit โดยใช้ argument `initial_layout` โปรดทราบว่า layout ของ Qubit ในตอนท้ายอาจแตกต่างจาก layout เริ่มต้น เนื่องจาก Transpiler อาจสลับ Qubit โดยใช้ swap gates หรือวิธีอื่นๆ

ในตัวอย่างด้านล่าง เราสร้าง initial layout สำหรับ [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) mock backend โดยการสร้าง object [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout) layout ของเรา map Qubit แรกของ Circuit ไปยัง Qubit 5 ของ Sherbrooke และ map Qubit ที่สองของ Circuit ไปยัง Qubit 6 ของ Sherbrooke โปรดทราบว่า physical qubit จะแสดงเป็น integer เสมอ

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

นอกจากการระบุ Layout object แล้ว คุณยังสามารถส่ง list ของ integer ได้ โดย element ที่ $i$ ของ list จะมี physical qubit ที่ Qubit ที่ $i$ ควร map ไปยัง ตัวอย่างเช่น:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

คุณสามารถใช้ฟังก์ชัน [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) เพื่อสร้างแผนภาพของ device graph พร้อมข้อมูลข้อผิดพลาดและ physical qubit ที่มีป้ายกำกับ คุณยังสามารถดูแผนภาพที่คล้ายกันได้ที่หน้า [Compute resources](https://quantum.cloud.ibm.com/computers)